In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [4]:
df.drop_duplicates(inplace=True)


In [5]:
df.shape


(49582, 2)

## Pre-processing

### 1. Converting to lowercase

In [6]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [7]:
import re

def remove_url(text):
    text = re.sub(r"http\S+", "", text)
    return text


df["review"]= df["review"].apply(remove_url)

### 3. Removing punctuations

In [8]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text)
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [9]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 4. Removing HTML

In [10]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing the Stopwords

In [31]:
import nltk

In [32]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [51]:
def remove_stopwords(text):
    token = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in token:
        if token in stop_words:
            text = text.replace(word, "")

    return text
df["review"] = df["review"].apply(remove_stopwords)

In [52]:
df.head()

,review,sentiment
0,one of the other review ha mention that after ...,1
1,a wonder littl product br br the film techniqu...,1
2,i thought thi wa a wonder way to spend time on...,1
3,basic there a famili where a littl boy jake th...,0
4,petter mattei love in the time of money is a v...,1


### 6. Stemming

In [53]:
from nltk.stem import PorterStemmer

In [54]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)
    

In [55]:
df.head()

,review,sentiment
0,one of the other review ha mention that after ...,1
1,a wonder littl product br br the film techniqu...,1
2,i thought thi wa a wonder way to spend time on...,1
3,basic there a famili where a littl boy jake th...,0
4,petter mattei love in the time of money is a v...,1


### 7. Encoding

In [56]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [57]:
y = df["sentiment"]

In [58]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [59]:
df.head()

,review,sentiment
0,one of the other review ha mention that after ...,1
1,a wonder littl product br br the film techniqu...,1
2,i thought thi wa a wonder way to spend time on...,1
3,basic there a famili where a littl boy jake th...,0
4,petter mattei love in the time of money is a v...,1


In [60]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

## Dataset & Data Loaders

In [61]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [62]:
X_train.shape

(39665, 5000)

In [63]:
X_test.shape

(9917, 5000)

In [64]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [65]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [66]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()    
)


test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [67]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN

In [68]:
import torch.nn as nn
import torch.optim as optim

In [73]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

        

In [74]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [75]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.13815633952617645
epoch = 2/10 and loss = 0.13736557960510254
epoch = 3/10 and loss = 0.24008101224899292
epoch = 4/10 and loss = 0.12001775950193405
epoch = 5/10 and loss = 0.3559582829475403
epoch = 6/10 and loss = 0.2542417645454407
epoch = 7/10 and loss = 0.21362915635108948
epoch = 8/10 and loss = 0.13806703686714172
epoch = 9/10 and loss = 0.2315140962600708
epoch = 10/10 and loss = 0.2645156979560852


In [76]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 87.37521427851165
